# Episode 33: Sparse autoencoder features inside our GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep033-sae.ipynb)

We train a sparse autoencoder on the residual stream of our web model, read what its features fire on, measure what it costs the model, and steer the model's writing with a feature.

[The video](https://youtu.be/34t5PYpZ1W0) · [The episode's page](https://mvgpt.nenuadrian.com/ep033-sae/) · [The docs](https://nenuadrian.github.io/mvGPT/sae/) · [The code at `ep033/end`](https://github.com/nenuadrian/mvGPT/tree/ep033/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep033/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain-web")

## The data
FineWeb-Edu's pages, which the autoencoder trains on, and the held-out pages for the rest.

In [ ]:
!python data.py --web 1  # FineWeb-Edu: two files of about 2 GB, the web merges, the held-out pages and a file of tokens

## How long the stream's vectors are
The test checks that `stream` gives what the model's blocks give. Then episode 19's web model,
its stream after block 6, position by position: the first token's vector comes out many times
longer than the rest, which looks like the attention sink.

In [ ]:
!python -m pytest -q tests/test_sae.py
!python sae.py norms checkpoints/pretrain-web/model.pt

## Training it
Fifteen minutes on the stream after block 6, with 12,288 features and 32 of them for each
token, to `checkpoints/sae/sae.pt`. Each line is a hundred steps: the share of the stream's
variance the autoencoder doesn't explain, and the share of features that haven't fired in two
million tokens.

In [ ]:
!python sae.py train checkpoints/pretrain-web/model.pt

## What it costs the model
The next-token loss on held-out pages with the model as it is, through the autoencoder after
block 6, and with the average token there instead: how much of the way back the autoencoder
gets.

In [ ]:
!python sae.py evaluate checkpoints/sae/sae.pt

## The video's autoencoder
Your autoencoder's features are numbered differently from the video's, so from here on it's
the video's, from the Hub, in `video/`. To read your own instead, put `checkpoints/sae/sae.pt`
in its place, and pick the features from what `features` prints.

In [ ]:
print(hf_hub_download("nenuadrian/mvGPT-checkpoints", "checkpoints/sae/sae.pt", local_dir="video"))

## The features on a sentence
The strongest features on each token, leaving out the few that fire on over half of all tokens.
In the video, 1621 fired on Paris, 10652 on capital, and 8287 on both Paris and France.

In [ ]:
!python sae.py features video/checkpoints/sae/sae.pt "The Eiffel Tower is in Paris, the capital of France."

## Six of them, on held-out pages
How often each fires, its strongest contexts, and the tokens it raises through the model's head:
1621 for a city's name, 10652 for capital, 1767 for carbon, which raises dioxide, and 5386 for
the moon.

In [ ]:
!python sae.py show video/checkpoints/sae/sae.pt 1621 10652 1767 --examples 4
!python sae.py show video/checkpoints/sae/sae.pt 5386 7516 6198 --examples 4

## Steering
The moon feature's direction added to the stream after block 6, at a strength of 12 and then 30,
each time next to the text as it is, from the same seed.

In [ ]:
!python sae.py steer video/checkpoints/sae/sae.pt 5386 "My favourite thing to do at night is" --strength 12 --tokens 45
!python sae.py steer video/checkpoints/sae/sae.pt 5386 "My favourite thing to do at night is" --strength 30 --tokens 45